# Exploratory Data Analysis — Sales Forecasting

**Objective:** Understand the dataset, identify data quality issues, and explore patterns before building forecasting models.

**Dataset:** `Forecasting Case- Study.xlsx` — US state-level beverage sales data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.size"] = 12

DATA_PATH = Path("../data/Forecasting Case- Study.xlsx")

## 1. Load and Inspect Raw Data

In [ ]:
df = pd.read_excel(DATA_PATH)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.tail(10)

## 2. Data Quality Checks

In [ ]:
# Missing values
print("=== Missing Values ===")
print(df.isnull().sum())
print(f"\nTotal nulls: {df.isnull().sum().sum()}")

# Duplicates
print(f"\n=== Duplicates ===")
dupes = df.duplicated().sum()
print(f"Duplicate rows: {dupes}")

# Negative or zero sales
print(f"\n=== Sales Value Checks ===")
print(f"Zero sales rows: {(df['Total'] == 0).sum()}")
print(f"Negative sales rows: {(df['Total'] < 0).sum()}")

In [ ]:
# Unique values per column
print("=== Unique Values ===")
for col in df.columns:
    print(f"{col}: {df[col].nunique()}")

print(f"\n=== Category Column ===")
print(df["Category"].value_counts())
print("\nCategory has only 1 unique value — can be dropped.")

## 3. Date Analysis

In [ ]:
# Parse dates — mixed format in the raw data
df["Date"] = pd.to_datetime(df["Date"], format="mixed", dayfirst=True)

print(f"Date range: {df['Date'].min()} → {df['Date'].max()}")
print(f"Total span: {(df['Date'].max() - df['Date'].min()).days} days")
print(f"Unique dates: {df['Date'].nunique()}")
print(f"Rows per state: {df.groupby('State').size().unique()}")

In [ ]:
# Check date gaps for a single state to understand frequency
sample_state = df[df["State"] == "California"].sort_values("Date").copy()
sample_state["gap_days"] = sample_state["Date"].diff().dt.days

print("=== Date Gap Distribution (California) ===")
print(sample_state["gap_days"].describe())
print(f"\n=== Gap Value Counts ===")
print(sample_state["gap_days"].value_counts().sort_index())

In [ ]:
# Visualize date gaps across all states
all_gaps = []
for state in df["State"].unique():
    state_df = df[df["State"] == state].sort_values("Date")
    gaps = state_df["Date"].diff().dt.days.dropna()
    all_gaps.extend(gaps.values)

plt.figure(figsize=(12, 5))
plt.hist(all_gaps, bins=50, edgecolor="black", alpha=0.7)
plt.xlabel("Gap Between Consecutive Dates (days)")
plt.ylabel("Frequency")
plt.title("Distribution of Date Gaps Across All States")
plt.axvline(x=7, color="red", linestyle="--", label="7 days (weekly)")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Most common gap: {pd.Series(all_gaps).mode().values[0]} days")
print(f"Gaps > 14 days: {sum(1 for g in all_gaps if g > 14)} occurrences")

## 4. Sales Distribution Analysis

In [ ]:
# Overall sales distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df["Total"], bins=50, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Sales (Total)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Sales Distribution")

axes[1].hist(np.log1p(df["Total"]), bins=50, edgecolor="black", alpha=0.7, color="orange")
axes[1].set_xlabel("Log(Sales)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Log-Transformed Sales Distribution")

plt.tight_layout()
plt.show()

In [ ]:
# Sales by state — which states have the highest/lowest sales
state_sales = df.groupby("State")["Total"].mean().sort_values(ascending=False)

plt.figure(figsize=(14, 8))
state_sales.plot(kind="barh", color="steelblue", edgecolor="black")
plt.xlabel("Average Sales")
plt.ylabel("State")
plt.title("Average Sales by State")
plt.tight_layout()
plt.show()

print("Top 5 states by avg sales:")
print(state_sales.head())
print("\nBottom 5 states by avg sales:")
print(state_sales.tail())

In [ ]:
# Box plot — sales spread per state
plt.figure(figsize=(16, 8))
state_order = df.groupby("State")["Total"].median().sort_values(ascending=False).index
sns.boxplot(data=df, x="Total", y="State", order=state_order, color="lightblue")
plt.title("Sales Distribution by State")
plt.xlabel("Sales (Total)")
plt.tight_layout()
plt.show()

## 5. Time Series Trends

In [ ]:
# Aggregate sales across all states over time
total_by_date = df.groupby("Date")["Total"].sum().sort_index()

plt.figure(figsize=(14, 6))
plt.plot(total_by_date.index, total_by_date.values, linewidth=1.5, color="steelblue")
plt.xlabel("Date")
plt.ylabel("Total Sales (All States)")
plt.title("Overall Sales Trend Over Time")
plt.tight_layout()
plt.show()

In [ ]:
# Time series for top 5 states
top_states = df.groupby("State")["Total"].mean().nlargest(5).index.tolist()

fig, axes = plt.subplots(5, 1, figsize=(14, 16), sharex=True)
for i, state in enumerate(top_states):
    state_data = df[df["State"] == state].sort_values("Date")
    axes[i].plot(state_data["Date"], state_data["Total"], linewidth=1.2)
    axes[i].set_title(state)
    axes[i].set_ylabel("Sales")

plt.xlabel("Date")
plt.suptitle("Sales Trend — Top 5 States by Average Sales", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Time series for bottom 5 states
bottom_states = df.groupby("State")["Total"].mean().nsmallest(5).index.tolist()

fig, axes = plt.subplots(5, 1, figsize=(14, 16), sharex=True)
for i, state in enumerate(bottom_states):
    state_data = df[df["State"] == state].sort_values("Date")
    axes[i].plot(state_data["Date"], state_data["Total"], linewidth=1.2, color="coral")
    axes[i].set_title(state)
    axes[i].set_ylabel("Sales")

plt.xlabel("Date")
plt.suptitle("Sales Trend — Bottom 5 States by Average Sales", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 6. Seasonality Analysis

In [ ]:
# Monthly seasonality — average sales by month across all states
df["month"] = df["Date"].dt.month
df["year"] = df["Date"].dt.year
df["day_of_week"] = df["Date"].dt.dayofweek

monthly_avg = df.groupby("month")["Total"].mean()

plt.figure(figsize=(10, 5))
monthly_avg.plot(kind="bar", color="steelblue", edgecolor="black")
plt.xlabel("Month")
plt.ylabel("Average Sales")
plt.title("Average Sales by Month (Seasonality Check)")
plt.xticks(range(12), ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"], rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Year-over-year trend
yearly_avg = df.groupby("year")["Total"].mean()

plt.figure(figsize=(8, 5))
yearly_avg.plot(kind="bar", color="teal", edgecolor="black")
plt.xlabel("Year")
plt.ylabel("Average Sales")
plt.title("Average Sales by Year (Trend Check)")
plt.tight_layout()
plt.show()

In [ ]:
# Monthly trend by year — heatmap
pivot = df.groupby(["year", "month"])["Total"].mean().unstack()

plt.figure(figsize=(12, 5))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="YlOrRd", linewidths=0.5)
plt.title("Average Sales — Year vs Month Heatmap")
plt.xlabel("Month")
plt.ylabel("Year")
plt.tight_layout()
plt.show()

In [ ]:
# Day of week distribution (what day are records on?)
dow_counts = df["day_of_week"].value_counts().sort_index()
dow_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

plt.figure(figsize=(8, 4))
plt.bar(dow_labels, dow_counts.values, color="mediumpurple", edgecolor="black")
plt.xlabel("Day of Week")
plt.ylabel("Number of Records")
plt.title("Records by Day of Week")
plt.tight_layout()
plt.show()

print("Most data points fall on specific days — data may already be weekly snapshots.")

## 7. Outlier Detection

In [ ]:
# IQR-based outlier detection per state
outlier_counts = {}
for state in df["State"].unique():
    state_data = df[df["State"] == state]["Total"]
    q1 = state_data.quantile(0.25)
    q3 = state_data.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_outliers = ((state_data < lower) | (state_data > upper)).sum()
    if n_outliers > 0:
        outlier_counts[state] = n_outliers

if outlier_counts:
    print(f"States with outliers (IQR method): {len(outlier_counts)}")
    outlier_df = pd.Series(outlier_counts).sort_values(ascending=False)
    print(outlier_df)
else:
    print("No outliers detected using IQR method.")

## 8. Correlation Between States

In [ ]:
# Pivot to wide format and check correlation between states
wide = df.pivot_table(index="Date", columns="State", values="Total")
corr = wide.corr()

plt.figure(figsize=(16, 14))
sns.heatmap(corr, cmap="coolwarm", center=0, vmin=-1, vmax=1, linewidths=0.3, fmt=".1f")
plt.title("Sales Correlation Between States")
plt.tight_layout()
plt.show()

print(f"Average inter-state correlation: {corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack().mean():.3f}")

## 9. Stationarity Check

In [ ]:
# Rolling mean and std for a sample state — visual stationarity check
sample = df[df["State"] == "California"].sort_values("Date").set_index("Date")["Total"]

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Rolling mean
axes[0].plot(sample.index, sample.values, label="Actual", alpha=0.6)
axes[0].plot(sample.rolling(window=12).mean(), label="Rolling Mean (12w)", color="red", linewidth=2)
axes[0].set_title("California — Sales with Rolling Mean")
axes[0].legend()

# Rolling std
axes[1].plot(sample.rolling(window=12).std(), label="Rolling Std (12w)", color="orange", linewidth=2)
axes[1].set_title("California — Rolling Standard Deviation")
axes[1].legend()

plt.tight_layout()
plt.show()

print("If rolling mean/std change over time, the series is non-stationary.")
print("ARIMA/SARIMA will need differencing. Prophet handles this internally.")

## 10. Key Findings

**Data overview:**
- 8,084 rows — 43 states, 188 records each, single category (Beverages)
- Date range: Jan 2019 → Dec 2023 (~5 years)
- No null values, but date gaps are irregular (not clean weekly intervals)

**Issues to handle in preprocessing:**
- Mixed date formats — need `format='mixed'` parsing
- Irregular time gaps — need resampling to consistent weekly frequency
- Missing weeks after resampling — need forward-fill or interpolation
- Category column is useless (only "Beverages") — drop it

**Patterns observed:**
- Clear seasonal patterns visible in monthly aggregation
- Year-over-year trend present — sales may be growing/shifting
- States are highly correlated — similar macro patterns
- Some states have outliers worth monitoring but not removing

**Implications for modeling:**
- ARIMA/SARIMA: will need differencing if series is non-stationary
- Prophet: handles trend + seasonality natively
- XGBoost: will benefit heavily from lag/rolling/calendar features
- LSTM: feed windowed sequences, keep architecture simple